This notebook is for generating "silver label" examples using the trained span identification and technique classification models in order to train a lighter weight model. The raw news article data pre-adding silver labels is from the English-only subset of the Common Crawl News dataset. Once run through the existing models to get "silver labels," we use these examples to train a xx model to be used in our Chrome extension.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path
from datasets import load_dataset
import os
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification
import json
from tqdm.auto import tqdm
from google import genai
from google.genai import types
from dotenv import load_dotenv
import time

In [ ]:
#Identify base directory to ensure portability
BASE_DIR = Path.cwd().resolve().parent
MODELS_DIR = BASE_DIR / "models"
interim_dir = BASE_DIR / "data" / "interim"
interim_dir.mkdir(parents=True, exist_ok=True)
output_file = interim_dir / "news_with_labels.csv"
second_output_file = Path("../data/interim/news_with_second_opinion.csv")
DATA_PATH = BASE_DIR / "data" / "processed" / "semeval_tc_cleaned.csv"

SI_DIR = MODELS_DIR / "semeval_roberta_scanner"
SI_SPEC_DIR = MODELS_DIR / "semeval_roberta_scanner_specialist"
TC_DIR = MODELS_DIR / "semeval_roberta_classifier"

SI_MODEL_PATH = f"{os.fspath(SI_DIR.absolute())}"
SI_SPEC_PATH = f"{os.fspath(SI_SPEC_DIR.absolute())}"
TC_MODEL_PATH = f"{os.fspath(TC_DIR.absolute())}"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
#Run training notebooks if models are missing
REQUIRED_FILES = ["config.json", "model.safetensors"]

def model_exists(path):
    path = Path(path)
    has_weights = any(path.glob("*.bin")) or any(path.glob("*.safetensors"))
    return has_weights

if not model_exists(SI_MODEL_PATH):
    print("SI Model missing. Running training notebook...")
    %run 4.1-fp-semeval-si-modeling.ipynb
if not model_exists(TC_MODEL_PATH):
    print("TC Model missing. Running training notebook...")
    %run 4.2-fp-semeval-tc-modeling.ipynb

In [ ]:
#Load Base SI Model (RoBERTa token-classifier for span detection)
print(f"Loading Base SI Model from: {SI_MODEL_PATH}...")
si_tokenizer = AutoTokenizer.from_pretrained(SI_MODEL_PATH)
si_model = AutoModelForTokenClassification.from_pretrained(SI_MODEL_PATH, local_files_only=True).to(device)
si_model.eval()

#Load Specialist SI Model
print(f"Loading Specialist SI Model from: {SI_SPEC_PATH}...")
si_spec_model = AutoModelForTokenClassification.from_pretrained(SI_SPEC_PATH, local_files_only=True).to(device)
si_spec_model.eval()

In [ ]:
#Load TC Model (Technique Classification)
tc_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
tc_model = AutoModelForSequenceClassification.from_pretrained(TC_MODEL_PATH).to(device)
tc_model.eval()

In [ ]:
#Set thresholds for each technique
OPTIMIZED_THRESHOLDS = {
    'Appeal_to_Authority': 0.60,
    'Appeal_to_fear-prejudice': 0.50,
    'Bandwagon_Reductio_ad_hitlerum': 0.10,
    'Black-and-White_Fallacy': 0.20,
    'Causal_Oversimplification': 0.20,
    'Doubt': 0.35,
    'Exaggeration_Minimisation': 0.40,
    'Flag-Waving': 0.45,
    'Loaded_Language': 0.40,
    'Name_Calling_Labeling': 0.55,
    'Repetition': 0.40,
    'Slogans': 0.30,
    'Thought-terminating_Cliches': 0.15,
    'Whataboutism_Straw_Men_Red_Herring': 0.15
}

In [ ]:
def run_pipeline_batched(texts):
    """
    Processes a list of texts through the SI -> Cascade -> TC pipeline.
    """
    #1. SI Tokenization (Batch)
    inputs = si_tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        return_offsets_mapping=True
    ).to(device)

    offsets_batch = inputs.pop("offset_mapping")

    with torch.no_grad():
        #2. SI Model Inference (Batch)
        base_outputs = si_model(**inputs)
        base_preds_batch = torch.argmax(base_outputs.logits, dim=-1)

        # 3. Specialist Model Inference (Batch)
        spec_outputs = si_spec_model(**inputs)
        spec_probs_batch = F.softmax(spec_outputs.logits, dim=-1)
        propaganda_prob_batch = spec_probs_batch[:, :, 1]

    #4. Apply Cascade Logic & Extract Spans for each item in batch
    batch_final_results = []

    for i in range(len(texts)):
        text = texts[i]
        base_preds = base_preds_batch[i]
        prop_probs = propaganda_prob_batch[i]
        offsets = offsets_batch[i]

        #Merge predictions
        final_preds = base_preds.clone()
        mask = (base_preds == 0) & (prop_probs > 0.5)
        final_preds[mask] = 1

        #Extract spans
        predicted_spans = []
        current_span = None
        for j, pred in enumerate(final_preds):
            label = pred.item()
            start, end = offsets[j]
            if start == end: continue
            if label in [1, 2]:
                if current_span is None:
                    current_span = [start.item(), end.item()]
                else:
                    current_span[1] = end.item()
            elif current_span:
                predicted_spans.append(tuple(current_span))
                current_span = None
        if current_span: predicted_spans.append(tuple(current_span))

        #5. Technique Classification (TC) for extracted spans
        article_results = []
        for span in predicted_spans:
            span_text = text[span[0]:span[1]].strip()
            if not span_text: continue

            tc_inputs = tc_tokenizer(span_text, return_tensors="pt", truncation=True, padding=True).to(device)
            with torch.no_grad():
                tc_logits = tc_model(**tc_inputs).logits
                probs = torch.sigmoid(tc_logits)[0]

            found_techniques = []
            for class_id, prob in enumerate(probs):
                tech_name = tc_model.config.id2label[class_id]
                if prob.item() >= OPTIMIZED_THRESHOLDS.get(tech_name, 0.5):
                    found_techniques.append(tech_name)

            if not found_techniques:
                found_techniques.append(tc_model.config.id2label[torch.argmax(probs).item()])

            for tech in found_techniques:
                article_results.append({"span": tuple(span), "technique": tech})

        batch_final_results.append(article_results)

    return batch_final_results

In [ ]:
#Load `news_with_labels.csv` if it already exists; otherwise, run the labeling pipeline and save the result
if output_file.exists():
    news = pd.read_csv(output_file, names=['text', 'propaganda'], header=0)
    news["propaganda"] = news["propaganda"].apply(json.loads)
    display(news)
else:
    print("Cache not found. Downloading the data and running the RoBERTa labeling pipeline (this will take time)...")
    #Load the news article dataset
    news = load_dataset("vblagoje/cc_news", split="train")
    news = news.to_pandas()

    #Constrain to only the text, as that's the only input our extension will be given
    #And take a random sample of articles because the dataset is unreasonably large
    news = news[['text']].sample(frac=0.06, random_state=42).reset_index(drop=True)
    display(news)

    #Use run pipeline function to get predicted propaganda spans and labels from all the text
    print("Running pipeline...")
    BATCH_SIZE = 32
    all_predictions = []
    texts_to_process = news['text'].tolist()

    for i in tqdm(range(0, len(texts_to_process), BATCH_SIZE)):
        batch = texts_to_process[i : i + BATCH_SIZE]
        batch_results = run_pipeline_batched(batch)
        all_predictions.extend(batch_results)

    news['propaganda'] = all_predictions
    display(news)

    #Save the dataframe so it can be reused later
    print("Saving DataFrame...")
    news_to_save = news.copy()
    news_to_save["propaganda"] = news_to_save["propaganda"].apply(json.dumps)

    news_to_save.to_csv(output_file, index=False)
    print(f"Silver labels saved to {output_file}")

In [ ]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key, http_options=types.HttpOptions(api_version='v1'))

def get_second_opinion(text, spans):
    try:
        prompt = f"""
        I am identifying and classifying propaganda in an article.

        Here are the exact types of propaganda I'm looking for with their definitions and an example for each. Only look for and consider these specific techniques:
        1. Loaded_Language. Definition: Using specific words and phrases with strong emotional implications (either positive or negative) to influence an audience. Uses emotional words without a concrete argument. Example: Outrage as Donald Trump suggests injecting disinfectant to kill virus.
        2. Name_Calling_Labeling. Definition: Labeling the object of the propaganda campaign as either something the target audience fears, hates, finds undesirable or loves, praises. The object of the sentence with the main topic is a person or group with a different opinion, and they are being attacked or diminished. Example: Coronavirus emergency is ’Public Enemy Number 1’
        3. Repetition. Definition: Repeating the same message over and over again, so that the audience will eventually accept it. Uses emotional words to prove an argument. Example: I still have a dream. It is a dream deeply rooted in the American dream. I have a dream that one day
        4. Exaggeration_Minimisation. Definition: Either representing something in an excessive manner: making things larger, better, worse or making something seem less important or smaller than it actually is. Uses emotional words without a concrete argument. Example: Coronavirus ‘risk to the American people remains very low’, Trump said.
        5. Doubt. Definition: Questioning the credibility of someone or something. Uses emotional words to prove an argument. Example: Can the same be said for the Obama Administration?
        6. Appeal_to_fear-prejudice. Definition: Seeking to build support for an idea by instilling anxiety and/or panic in the population towards an alternative, possibly based on preconceived judgments. Uses emotional words to prove an argument. Example: A dark, impenetrable and “irreversible” winter of persecution of the faithful by their own shepherds will fall.
        7. Flag-Waving. Definition: Playing on strong national feeling (or with respect to any group, e.g., race, gender, political preference) to justify or promote an action or idea. Uses emotional words without a concrete argument. Example: Mueller attempts to stop the will of We the People!!! It’s time to jail Mueller.
        8. Causal_Oversimplification. Definition: Assuming a single cause or reason when there are multiple causes behind an issue. We include in the definition also scapegoating, e.g., transferring the blame to one person or group of people without investigating the complexities of an issue. Tries to simplify the problem by reducing a complex problem to only one cause. Example: If France had not have declared war on Germany then World War II would have never happened.
        9. Slogans. Definition: A brief and striking phrase that may include labeling and stereotyping. Slogans tend to act as emotional appeals. Uses emotional words to prove an argument. Example: “BUILD THE WALL!” Trump tweeted.
        10. Appeal_to_Authority. Definition: Stating that a claim is true simply because a valid authority or expert on the issue supports it, without any other supporting evidence. We include in this technique the special case in which the reference is not an authority or an expert, although it is referred to as testimonial in the literature. Tries to simplify the problem by showing support from a reference. Example: Monsignor Jean-Franois Lantheaume, who served as first Counsellor of the Nunciature in Washington, confirmed that “Vigan said the truth. That’s all.”
        11. Black-and-White_Fallacy. Definition: Presenting two alternative options as the only possibilities, when in fact more possibilities exist. Dictatorship is an extreme case: telling the audience exactly what actions to take, eliminating any other possible choice. Tries to simplify the problem by presenting only a subset of the options. Example: Everyone is guilty for the good he could have done and did not do . . . If we do not oppose evil, we tacitly feed it.
        12. Thought-terminating_Cliches. Definition: Words or phrases that discourage critical thought and meaningful discussion on a topic. They are typically short, generic sentences that offer seemingly simple answers to complex questions or that distract attention away from other lines of thought. Tries to simplify the problem by discouraging critical thought and meaningful discussion. Example: I do not really see any problems there. Marx is the President.
        13. Whataboutism_Straw_Men_Red_Herring. Definition: Here we merge together three techniques, which are relatively rare taken individually: (i) Whataboutism: Discredit an opponent’s position by charging them with hypocrisy without directly disproving their argument. (ii) Straw man: When an opponent’s proposition is substituted with a similar one, which is then refuted instead of the original. Weston (2000, p. 78) specifies the characteristics of the substituted proposition: “caricaturing anopposing view so that it is easy to refute”. (iii) Red herring: Introducing irrelevant material to the issue being discussed, so that everyone’s attention is diverted away from the points made. Adds irrelevant data or changes the argument. Example: You may claim that the death penalty is an ineffective deterrent against crime – but what about the victims of crime? How do you think surviving family members feel when they see the man who murdered their son kept in prison at their expense? Is right that they should pay for their son’s murderer to be fed and housed?
        14. Bandwagon_Reductio_ad_hitlerum. Definition: Here we merge together two techniques, which are relatively rare taken individually: (i) Bandwagon. Attempting to persuade the target audience to join in and take the course of action because “everyone else is taking the same action”. (ii) Reductio ad hitlerum: Persuading an audience to disapprove an action or idea by suggesting that it is popular with groups hated in contempt by the target audience. It can refer to any person or concept with a negative connotation. Object of the sentence with the main topic is a third figure, and the statement tries to persuade you based on them to approve or disapprove of something. Example: EU no longer considers #Hamas a terrorist group. Time for US to do same.

        My base model found that these character spans may contain the indicated propaganda techniques: {spans}
        From this article text: "{text}"

        Your Instructions:
        1. Review the spans found by the base model.
        2. If a span is not actually an example of one of these propaganda techniques, remove it.
        3. If the base model missed a clear propaganda technique or the span was somewhat off, add it with the correct span of characters where it appears.
        4. Provide the final list in JSON format: [[{{"span": [start, end], "technique": "name" text}}]] using technique names in the same format they originally appeared in (Doubt, Appeal_to_Authority, Repetition, Appeal_to_fear-prejudice, Slogans, Black-and-White_Fallacy, Loaded_Language, Flag-Waving, Name_Calling_Labeling, Causal_Oversimplification, Whataboutism_Straw_Men_Red_Herring, Exaggeration_Minimisation, Bandwagon_Reductio_ad_hitlerum, Thought-terminating_Cliches). IMPORTANT: Output ONLY a raw JSON array. Do not include markdown code blocks or explanations.
        """
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt)

        #Pause AFTER the request to ensure we stay under the 15 RPM limit
        time.sleep(4.5)
        return response.text

    except Exception as e:
        if "429" in str(e):
            print("Rate limit hit! Sleeping for 60 seconds...")
            time.sleep(60)
            return get_second_opinion(text, spans)
        else:
            print(f"Error: {e}")
            return None

In [ ]:
#If the output exists, see where we left off
if second_output_file.exists():
    processed_df = pd.read_csv(second_output_file)
    processed_indices = processed_df.index.tolist()
    start_idx = len(processed_indices)
    print(f"Resuming from index {start_idx}...")
else:
    #Initialize file with headers
    pd.DataFrame(columns=list(news.columns) + ['llm_opinion']).to_csv(second_output_file, index=False)
    start_idx = 0

#2. Run the loop
for i in tqdm(range(start_idx, len(news))):
    row = news.iloc[i]
    text = row['text']
    spans = row['propaganda']

    opinion = get_second_opinion(text, spans)

    if opinion:
        current_data = row.to_dict()
        current_data['llm_opinion'] = opinion

        batch_df = pd.DataFrame([current_data])
        batch_df.to_csv(output_file, mode='a', index=False, header=False)
    else:
        print(f"Skipping index {i} due to error.")

print("All second opinions gathered!")

In [ ]:
#Train/test/split
#Vectorize or otherwise transform text somehow
#Train models and compare performance
#Get some examples